# Phase 4: Temperature Scaling for Probability Calibration

**Objective**: Calibrate model probabilities to reflect real confidence (p=0.8 → 80% accuracy).

**Pipeline**:
1. Extract baseline predictions on val_calib
2. Optimize temperature T by minimizing NLL
3. Apply T on val_eval
4. Evaluate calibration improvement (ECE, Brier, NLL, Reliability Diagrams)

### ⚠️ APPLIED CORRECTIONS

**Identified Problem**: GroundingDINO returns **scores** (0-1), not raw logits.

**Implemented Solutions**:
1. ✅ Correct conversion: `logit = log(score / (1-score))` (inverse sigmoid)
2. ✅ Optimization over true logits
3. ✅ Corrected reliability diagrams
4. ✅ Calibration diagnosis before optimization
5. ✅ Verification checklist at the end

**What to Expect**:
- If T ≈ 1.0 → Model **was already calibrated** (good news!)
- If T > 1.0 → Model **overconfident** → Calibration reduces confidence
- If T < 1.0 → Model **underconfident** → Calibration increases confidence

## 1. Imports and Configuration

In [ ]:
print("kernel working")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import yaml
from scipy.optimize import minimize
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('./outputs/temperature_scaling')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'seed': 42,
    'iou_matching': 0.5,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'categories': ['person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle', 'traffic light', 'traffic sign']
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print(f"Device: {CONFIG['device']}")
print(f"Output: {OUTPUT_DIR}")

## 2. Load Baseline Model

In [ ]:
from groundingdino.util.inference import load_model, load_image, predict
from groundingdino.util import box_ops
from pathlib import Path

model_config = Path('/opt/program/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py')
model_weights = Path('/opt/program/GroundingDINO/weights/groundingdino_swint_ogc.pth')

model = load_model(str(model_config), str(model_weights))
model.to(CONFIG['device'])
model.eval()

TEXT_PROMPT = '. '.join(CONFIG['categories']) + '.'
print(f"Model loaded on {CONFIG['device']}")
print(f"Prompt: {TEXT_PROMPT}")

## 3. Inference on val_calib and Matching with GT

In [ ]:
from pycocotools.coco import COCO
import torchvision

def compute_iou(box1, box2):
    """Calculate IoU between two boxes [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def normalize_label(label):
    """Normalize model label to canonical category"""
    synonyms = {'bike': 'bicycle', 'motorbike': 'motorcycle', 'pedestrian': 'person', 
                'stop sign': 'traffic sign', 'red light': 'traffic light'}
    label_lower = label.lower().strip()
    if label_lower in synonyms:
        return synonyms[label_lower]
    for cat in CONFIG['categories']:
        if cat in label_lower:
            return cat
    return label_lower

def run_inference_on_image(model, image_path, text_prompt, conf_thresh, device):
    """Run inference on a single image"""
    image_source, image = load_image(str(image_path))
    boxes, scores, phrases = predict(model, image, text_prompt, conf_thresh, 0.25, device)
    
    if len(boxes) == 0:
        return []
    
    h, w = image_source.shape[:2]
    boxes_xyxy = box_ops.box_cxcywh_to_xyxy(boxes) * torch.tensor([w, h, w, h], dtype=torch.float32)
    
    detections = []
    for box, score, phrase in zip(boxes_xyxy.cpu().numpy(), scores.cpu().numpy(), phrases):
        cat = normalize_label(phrase)
        if cat in CONFIG['categories']:
            score_clipped = np.clip(float(score), 1e-7, 1 - 1e-7)
            logit = np.log(score_clipped / (1 - score_clipped))
            
            detections.append({
                'bbox': box.tolist(),
                'score': score_clipped,
                'logit': logit,
                'category': cat
            })
    
    if len(detections) == 0:
        return []
    
    boxes_t = torch.tensor([d['bbox'] for d in detections], dtype=torch.float32)
    scores_t = torch.tensor([d['score'] for d in detections], dtype=torch.float32)
    keep = torchvision.ops.nms(boxes_t, scores_t, 0.65)
    
    return [detections[i] for i in keep.numpy()]

def match_predictions_to_gt(predictions, gt_anns, iou_thresh):
    """Assign TP/FP to predictions based on GT matching"""
    gt_matched = set()
    results = []
    
    cat_to_id = {cat: i+1 for i, cat in enumerate(CONFIG['categories'])}
    id_to_cat = {v: k for k, v in cat_to_id.items()}
    
    for pred in predictions:
        pred_cat = pred['category']
        pred_box = pred['bbox']
        
        best_iou = 0
        best_gt_idx = -1
        
        for idx, gt in enumerate(gt_anns):
            if idx in gt_matched:
                continue
            
            gt_cat_id = gt['category_id']
            gt_cat = id_to_cat.get(gt_cat_id, '')
            
            if gt_cat != pred_cat:
                continue
            
            gt_box = gt['bbox']
            gt_box_xyxy = [gt_box[0], gt_box[1], gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]]
            iou = compute_iou(pred_box, gt_box_xyxy)
            
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx
        
        if best_iou >= iou_thresh:
            gt_matched.add(best_gt_idx)
            is_tp = 1
        else:
            is_tp = 0
        
        results.append({
            'logit': pred['logit'],
            'score': pred['score'],
            'category': pred_cat,
            'is_tp': is_tp,
            'iou': best_iou
        })
    
    return results

print("Inference and matching functions defined")


In [ ]:
val_calib_json = Path('../data/bdd100k_coco/val_calib.json')
image_dir = Path('../data/bdd100k/bdd100k/bdd100k/images/100k/val')

coco_calib = COCO(str(val_calib_json))
img_ids = coco_calib.getImgIds()

print(f"Processing {len(img_ids)} images from val_calib...")

calib_data = []

for img_id in tqdm(img_ids[:500]):
    img_info = coco_calib.loadImgs(img_id)[0]
    img_path = image_dir / img_info['file_name']
    
    if not img_path.exists():
        continue
    
    predictions = run_inference_on_image(model, img_path, TEXT_PROMPT, 0.25, CONFIG['device'])
    gt_anns = coco_calib.loadAnns(coco_calib.getAnnIds(imgIds=img_id))
    matched = match_predictions_to_gt(predictions, gt_anns, CONFIG['iou_matching'])
    calib_data.extend(matched)

calib_df = pd.DataFrame(calib_data)
calib_df.to_csv(OUTPUT_DIR / 'calib_detections.csv', index=False)

print(f"\nTotal detections: {len(calib_df)}")
print(f"TP: {calib_df['is_tp'].sum()}, FP: {len(calib_df) - calib_df['is_tp'].sum()}")
print(f"Saved to: {OUTPUT_DIR / 'calib_detections.csv'}")

### Quick Calibration Diagnosis

In [ ]:
calib_df = pd.read_csv(OUTPUT_DIR / 'calib_detections.csv')

print("="*60)
print("CALIBRATION DIAGNOSIS ON VAL_CALIB")
print("="*60)

scores = calib_df['score'].values
labels = calib_df['is_tp'].values

# Split into confidence bins
bins = np.linspace(0, 1, 11)
digitized = np.digitize(scores, bins) - 1

print(f"\nTotal detections: {len(calib_df)}")
print(f"TP: {labels.sum()} ({labels.mean()*100:.1f}%)")
print(f"FP: {len(labels) - labels.sum()} ({(1-labels.mean())*100:.1f}%)")

print(f"\nScore range: [{scores.min():.3f}, {scores.max():.3f}]")
print(f"Score mean: {scores.mean():.3f}")

print("\n" + "-"*60)
print(f"{'Bin':<15} {'Confidence':<15} {'Accuracy':<15} {'Gap':<15} {'Count'}")
print("-"*60)

total_gap = 0
for i in range(10):
    mask = digitized == i
    if mask.sum() > 0:
        conf = scores[mask].mean()
        acc = labels[mask].mean()
        gap = abs(conf - acc)
        total_gap += gap * mask.sum() / len(scores)
        print(f"[{bins[i]:.1f}-{bins[i+1]:.1f}]  {conf:>10.3f}     {acc:>10.3f}     {gap:>10.3f}     {mask.sum():>6}")

print("-"*60)
print(f"ECE (Expected Calibration Error): {total_gap:.4f}")
print("="*60)

if scores.mean() > labels.mean():
    print("\n⚠️  MODEL OVERCONFIDENT: confidence > accuracy")
    print(f"   → Expected T > 1 to reduce confidence")
else:
    print("\n✓ Model underconfident or well calibrated")
    print(f"  → Expected T ≈ 1 or T < 1")

## 4. Optimize Global Temperature

In [ ]:
calib_df = pd.read_csv(OUTPUT_DIR / 'calib_detections.csv')

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def nll_loss(T, logits, labels):
    """Negative Log-Likelihood with temperature T"""
    T = max(T, 0.01)
    probs = sigmoid(logits / T)
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    nll = -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))
    return nll

logits = calib_df['logit'].values
labels = calib_df['is_tp'].values

# NLL before calibration
nll_before = nll_loss(1.0, logits, labels)
print(f"NLL before (T=1.0): {nll_before:.4f}")

# Optimize T
result = minimize(lambda T: nll_loss(T, logits, labels), x0=1.0, bounds=[(0.01, 10.0)], method='L-BFGS-B')
T_optimal = result.x[0]
nll_after = result.fun

print(f"\nOptimal T: {T_optimal:.4f}")
print(f"NLL after: {nll_after:.4f}")
print(f"NLL improvement: {nll_before - nll_after:.4f}")

# Save temperature
with open(OUTPUT_DIR / 'temperature.json', 'w') as f:
    json.dump({'T_global': T_optimal, 'nll_before': nll_before, 'nll_after': nll_after}, f, indent=2)

print(f"\nTemperature saved to: {OUTPUT_DIR / 'temperature.json'}")

## 5. Evaluate on val_eval

In [ ]:
with open(OUTPUT_DIR / 'temperature.json', 'r') as f:
    temp_data = json.load(f)
    T_optimal = temp_data['T_global']

val_eval_json = Path('../data/bdd100k_coco/val_eval.json')
coco_eval = COCO(str(val_eval_json))
img_ids_eval = coco_eval.getImgIds()

print(f"Processing {len(img_ids_eval)} images from val_eval...")

eval_data = []

for img_id in tqdm(img_ids_eval):
    img_info = coco_eval.loadImgs(img_id)[0]
    img_path = image_dir / img_info['file_name']
    
    if not img_path.exists():
        continue
    
    predictions = run_inference_on_image(model, img_path, TEXT_PROMPT, 0.25, CONFIG['device'])
    gt_anns = coco_eval.loadAnns(coco_eval.getAnnIds(imgIds=img_id))
    matched = match_predictions_to_gt(predictions, gt_anns, CONFIG['iou_matching'])
    eval_data.extend(matched)

eval_df = pd.DataFrame(eval_data)
eval_df.to_csv(OUTPUT_DIR / 'eval_detections.csv', index=False)

print(f"\nTotal detections: {len(eval_df)}")
print(f"TP: {eval_df['is_tp'].sum()}, FP: {len(eval_df) - eval_df['is_tp'].sum()}")
print(f"Saved to: {OUTPUT_DIR / 'eval_detections.csv'}")

## 6. Calculate Calibration Metrics

In [ ]:
def compute_calibration_metrics(logits, labels, T=1.0, n_bins=10):
    """Calculate ECE, Brier and NLL"""
    probs = sigmoid(logits / T)
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    
    nll = -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))
    brier = np.mean((probs - labels) ** 2)
    
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0.0
    bin_data = []
    
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (probs >= bin_lower) & (probs < bin_upper)
        prop_in_bin = np.mean(in_bin)
        
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(labels[in_bin])
            avg_confidence_in_bin = np.mean(probs[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            
            bin_data.append({
                'bin_lower': bin_lower,
                'bin_upper': bin_upper,
                'confidence': avg_confidence_in_bin,
                'accuracy': accuracy_in_bin,
                'count': np.sum(in_bin)
            })
        else:
            bin_data.append({
                'bin_lower': bin_lower,
                'bin_upper': bin_upper,
                'confidence': (bin_lower + bin_upper) / 2,
                'accuracy': 0,
                'count': 0
            })
    
    return {
        'nll': nll,
        'brier': brier,
        'ece': ece,
        'bin_data': bin_data
    }

print("Calibration functions defined")

In [ ]:
eval_df = pd.read_csv(OUTPUT_DIR / 'eval_detections.csv')
with open(OUTPUT_DIR / 'temperature.json', 'r') as f:
    T_optimal = json.load(f)['T_global']

logits_eval = eval_df['logit'].values
labels_eval = eval_df['is_tp'].values

# Metrics BEFORE calibration
metrics_before = compute_calibration_metrics(logits_eval, labels_eval, T=1.0)

# Metrics AFTER calibration
metrics_after = compute_calibration_metrics(logits_eval, labels_eval, T=T_optimal)

print("="*60)
print("CALIBRATION METRICS ON VAL_EVAL")
print("="*60)
print(f"\n{'Metric':<15} {'Before':<15} {'After':<15} {'Improvement':<15}")
print("-"*60)
print(f"{'NLL':<15} {metrics_before['nll']:<15.4f} {metrics_after['nll']:<15.4f} {metrics_before['nll']-metrics_after['nll']:<15.4f}")
print(f"{'Brier Score':<15} {metrics_before['brier']:<15.4f} {metrics_after['brier']:<15.4f} {metrics_before['brier']-metrics_after['brier']:<15.4f}")
print(f"{'ECE':<15} {metrics_before['ece']:<15.4f} {metrics_after['ece']:<15.4f} {metrics_before['ece']-metrics_after['ece']:<15.4f}")
print("="*60)

# Save metrics
results = {
    'T_optimal': T_optimal,
    'val_eval': {
        'before': {k: v for k, v in metrics_before.items() if k != 'bin_data'},
        'after': {k: v for k, v in metrics_after.items() if k != 'bin_data'}
    }
}

with open(OUTPUT_DIR / 'calibration_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nMetrics saved to: {OUTPUT_DIR / 'calibration_metrics.json'}")

## 7. Reliability Diagrams

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (T, title) in enumerate([(1.0, 'Before Calibration'), (T_optimal, 'After Calibration')]):
    metrics = compute_calibration_metrics(logits_eval, labels_eval, T=T)
    bin_data = metrics['bin_data']
    
    confidences = [b['confidence'] for b in bin_data if b['count'] > 0]
    accuracies = [b['accuracy'] for b in bin_data if b['count'] > 0]
    counts = [b['count'] for b in bin_data if b['count'] > 0]
    
    ax = axes[idx]
    
    ax.bar(range(len(confidences)), accuracies, width=0.8, alpha=0.5, color='skyblue', label='Accuracy', edgecolor='blue')
    ax.plot(range(len(confidences)), confidences, 'ro-', label='Confidence', markersize=8, linewidth=2)
    ax.plot(range(len(confidences)), np.linspace(0.05, 0.95, len(confidences)), 'k--', alpha=0.5, label='Perfect Calibration', linewidth=2)
    
    ax.set_xlabel('Confidence Bin', fontsize=12)
    ax.set_ylabel('Proportion', fontsize=12)
    ax.set_title(f'{title}\nECE = {metrics["ece"]:.4f}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(alpha=0.3, axis='y')
    ax.set_ylim([0, 1.05])
    ax.set_xticks(range(len(confidences)))
    ax.set_xticklabels([f'{i*0.1:.1f}' for i in range(len(confidences))], fontsize=9)
    
    for i, count in enumerate(counts):
        ax.text(i, -0.12, f'n={count}', ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to: {OUTPUT_DIR / 'reliability_diagram.png'}")

## 8. (Optional) Temperature Scaling per Class

In [ ]:
calib_df = pd.read_csv(OUTPUT_DIR / 'calib_detections.csv')

T_per_class = {}
min_samples = 50

for cat in CONFIG['categories']:
    cat_data = calib_df[calib_df['category'] == cat]
    
    if len(cat_data) < min_samples:
        T_per_class[cat] = T_optimal
        print(f"{cat:<20} - Insufficient samples ({len(cat_data)}), using global T: {T_optimal:.4f}")
        continue
    
    logits_cat = cat_data['logit'].values
    labels_cat = cat_data['is_tp'].values
    
    result = minimize(lambda T: nll_loss(T, logits_cat, labels_cat), x0=1.0, bounds=[(0.01, 10.0)], method='L-BFGS-B')
    T_cat = result.x[0]
    T_per_class[cat] = T_cat
    
    print(f"{cat:<20} - T: {T_cat:.4f} (n={len(cat_data)})")

with open(OUTPUT_DIR / 'temperature_per_class.json', 'w') as f:
    json.dump({'T_global': T_optimal, 'T_per_class': T_per_class}, f, indent=2)

print(f"\nPer-class temperatures saved to: {OUTPUT_DIR / 'temperature_per_class.json'}")

In [ ]:
eval_df = pd.read_csv(OUTPUT_DIR / 'eval_detections.csv')

with open(OUTPUT_DIR / 'temperature_per_class.json', 'r') as f:
    T_data = json.load(f)
    T_per_class = T_data['T_per_class']

eval_df['T_class'] = eval_df['category'].map(T_per_class)
eval_df['prob_calibrated_class'] = sigmoid(eval_df['logit'].values / eval_df['T_class'].values)

logits_eval = eval_df['logit'].values
labels_eval = eval_df['is_tp'].values
T_class_array = eval_df['T_class'].values

metrics_class = compute_calibration_metrics(logits_eval, labels_eval, T=1.0)
probs_class = sigmoid(logits_eval / T_class_array)
probs_class = np.clip(probs_class, 1e-7, 1 - 1e-7)
nll_class = -np.mean(labels_eval * np.log(probs_class) + (1 - labels_eval) * np.log(1 - probs_class))
brier_class = np.mean((probs_class - labels_eval) ** 2)

# ECE with T per class
bin_boundaries = np.linspace(0, 1, 11)
ece_class = 0.0
for i in range(len(bin_boundaries) - 1):
    in_bin = (probs_class >= bin_boundaries[i]) & (probs_class < bin_boundaries[i+1])
    if np.sum(in_bin) > 0:
        acc = np.mean(labels_eval[in_bin])
        conf = np.mean(probs_class[in_bin])
        ece_class += np.abs(acc - conf) * np.mean(in_bin)

print("="*60)
print("COMPARISON: Global T vs Per-Class T")
print("="*60)
print(f"\n{'Metric':<15} {'Global T':<15} {'Per-Class T':<15} {'Difference':<15}")
print("-"*60)
print(f"{'NLL':<15} {metrics_after['nll']:<15.4f} {nll_class:<15.4f} {metrics_after['nll']-nll_class:<15.4f}")
print(f"{'Brier Score':<15} {metrics_after['brier']:<15.4f} {brier_class:<15.4f} {metrics_after['brier']-brier_class:<15.4f}")
print(f"{'ECE':<15} {metrics_after['ece']:<15.4f} {ece_class:<15.4f} {metrics_after['ece']-ece_class:<15.4f}")
print("="*60)

## 9. Verify Impact on mAP (Detection)

In [ ]:
from pycocotools.cocoeval import COCOeval

def create_coco_results(image_dir, coco_gt, text_prompt, model, device, T=1.0, use_calibrated=True):
    """Generate results in COCO format with calibrated scores"""
    results = []
    cat_to_id = {cat: i+1 for i, cat in enumerate(CONFIG['categories'])}
    
    for img_id in tqdm(coco_gt.getImgIds()):
        img_info = coco_gt.loadImgs(img_id)[0]
        img_path = image_dir / img_info['file_name']
        
        if not img_path.exists():
            continue
        
        predictions = run_inference_on_image(model, img_path, text_prompt, 0.25, device)
        
        for pred in predictions:
            if use_calibrated:
                score_calibrated = sigmoid(pred['logit'] / T)
            else:
                score_calibrated = pred['score']
            
            results.append({
                'image_id': img_id,
                'category_id': cat_to_id.get(pred['category'], 1),
                'bbox': [pred['bbox'][0], pred['bbox'][1], 
                        pred['bbox'][2] - pred['bbox'][0], 
                        pred['bbox'][3] - pred['bbox'][1]],
                'score': float(score_calibrated)
            })
    
    return results

print("Calculating mAP BEFORE calibration...")
results_before = create_coco_results(image_dir, coco_eval, TEXT_PROMPT, model, CONFIG['device'], T=1.0, use_calibrated=False)

if len(results_before) > 0:
    coco_dt_before = coco_eval.loadRes(results_before)
    coco_eval_obj_before = COCOeval(coco_eval, coco_dt_before, 'bbox')
    coco_eval_obj_before.evaluate()
    coco_eval_obj_before.accumulate()
    print("\n--- mAP BEFORE ---")
    coco_eval_obj_before.summarize()
    map_before = coco_eval_obj_before.stats[0]
else:
    map_before = 0.0

print("\n\nCalculating mAP AFTER calibration...")
results_after = create_coco_results(image_dir, coco_eval, TEXT_PROMPT, model, CONFIG['device'], T=T_optimal, use_calibrated=True)

if len(results_after) > 0:
    coco_dt_after = coco_eval.loadRes(results_after)
    coco_eval_obj_after = COCOeval(coco_eval, coco_dt_after, 'bbox')
    coco_eval_obj_after.evaluate()
    coco_eval_obj_after.accumulate()
    print("\n--- mAP AFTER ---")
    coco_eval_obj_after.summarize()
    map_after = coco_eval_obj_after.stats[0]
else:
    map_after = 0.0

print(f"\n{'='*60}")
print(f"mAP before:  {map_before:.4f}")
print(f"mAP after:   {map_after:.4f}")
print(f"Difference:  {map_after - map_before:.4f}")
print(f"{'='*60}")
print("\nNote: Detection ranking does NOT change (only probability scale)")
print("Therefore mAP should remain practically the same.")

## 10. Confidence Distribution Analysis

In [ ]:
eval_df = pd.read_csv(OUTPUT_DIR / 'eval_detections.csv')

prob_before = sigmoid(eval_df['logit'].values)
prob_after = sigmoid(eval_df['logit'].values / T_optimal)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before distribution
axes[0].hist(prob_before[eval_df['is_tp']==1], bins=30, alpha=0.6, label='TP', color='green', density=True)
axes[0].hist(prob_before[eval_df['is_tp']==0], bins=30, alpha=0.6, label='FP', color='red', density=True)
axes[0].set_xlabel('Confidence', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Before Calibration', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# After distribution
axes[1].hist(prob_after[eval_df['is_tp']==1], bins=30, alpha=0.6, label='TP', color='green', density=True)
axes[1].hist(prob_after[eval_df['is_tp']==0], bins=30, alpha=0.6, label='FP', color='red', density=True)
axes[1].set_xlabel('Confidence', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title(f'After Calibration (T={T_optimal:.3f})', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to: {OUTPUT_DIR / 'confidence_distribution.png'}")

## 11. Risk-Coverage Curves

In [ ]:
def compute_risk_coverage(scores, labels):
    """Calculate risk-coverage curve ordering by descending confidence"""
    order = np.argsort(scores)[::-1]
    scores_sorted = scores[order]
    labels_sorted = labels[order]
    
    coverages = []
    risks = []
    
    for i in range(1, len(scores_sorted) + 1):
        coverage = i / len(scores_sorted)
        risk = 1 - np.mean(labels_sorted[:i])
        coverages.append(coverage)
        risks.append(risk)
    
    return coverages, risks

eval_df = pd.read_csv(OUTPUT_DIR / 'eval_detections.csv')

scores_before = sigmoid(eval_df['logit'].values)
labels = eval_df['is_tp'].values
cov_before, risk_before = compute_risk_coverage(scores_before, labels)

scores_after = sigmoid(eval_df['logit'].values / T_optimal)
cov_after, risk_after = compute_risk_coverage(scores_after, labels)

plt.figure(figsize=(10, 6))
plt.plot(cov_before, risk_before, 'b-', label='Before (T=1.0)', linewidth=2)
plt.plot(cov_after, risk_after, 'r-', label=f'After (T={T_optimal:.3f})', linewidth=2)
plt.xlabel('Coverage (fraction of retained predictions)', fontsize=12)
plt.ylabel('Risk (error rate)', fontsize=12)
plt.title('Risk-Coverage Trade-off', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'risk_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to: {OUTPUT_DIR / 'risk_coverage.png'}")

auc_before = np.trapz(risk_before, cov_before)
auc_after = np.trapz(risk_after, cov_after)
print(f"\nAUC Risk-Coverage before: {auc_before:.4f}")
print(f"AUC Risk-Coverage after:  {auc_after:.4f}")
print(f"Improvement (lower is better): {auc_before - auc_after:.4f}")

## 12. Final Report and Summary

In [ ]:
with open(OUTPUT_DIR / 'calibration_metrics.json', 'r') as f:
    metrics = json.load(f)

report = f"""
{'='*70}
PHASE 4: TEMPERATURE SCALING - FINAL REPORT
{'='*70}

1. OPTIMAL TEMPERATURE
   - Global T: {T_optimal:.4f}

2. CALIBRATION METRICS ON VAL_EVAL
   
   Metric         | Before (T=1.0) | After          | Improvement
   ----------------|----------------|----------------|---------------
   NLL             | {metrics_before['nll']:14.4f} | {metrics_after['nll']:14.4f} | {metrics_before['nll']-metrics_after['nll']:14.4f}
   Brier Score     | {metrics_before['brier']:14.4f} | {metrics_after['brier']:14.4f} | {metrics_before['brier']-metrics_after['brier']:14.4f}
   ECE             | {metrics_before['ece']:14.4f} | {metrics_after['ece']:14.4f} | {metrics_before['ece']-metrics_after['ece']:14.4f}

3. INTERPRETATION
   - T > 1: Overconfident model → Temperature Scaling reduces confidence
   - T < 1: Underconfident model → Temperature Scaling increases confidence
   - T ≈ 1: Model already well calibrated
   
   In this case: T = {T_optimal:.4f}
   {'→ Model was OVERCONFIDENT' if T_optimal > 1 else '→ Model was UNDERCONFIDENT' if T_optimal < 1 else '→ Model was already calibrated'}

4. IMPACT ON DETECTION
   - mAP before:  {map_before:.4f}
   - mAP after:   {map_after:.4f}
   - Difference:  {map_after - map_before:+.4f}
   
   → Calibration {'IMPROVES' if map_after >= map_before else 'SLIGHTLY REDUCES'} detection performance

5. GENERATED ARTIFACTS
   ✓ {OUTPUT_DIR / 'temperature.json'}
   ✓ {OUTPUT_DIR / 'calib_detections.csv'}
   ✓ {OUTPUT_DIR / 'eval_detections.csv'}
   ✓ {OUTPUT_DIR / 'calibration_metrics.json'}
   ✓ {OUTPUT_DIR / 'reliability_diagram.png'}
   ✓ {OUTPUT_DIR / 'confidence_distribution.png'}
   ✓ {OUTPUT_DIR / 'risk_coverage.png'}
   {'✓ ' + str(OUTPUT_DIR / 'temperature_per_class.json') if (OUTPUT_DIR / 'temperature_per_class.json').exists() else ''}

6. CONCLUSIONS FOR ADAS
   - Calibrated probabilities are more consistent with reality
   - When the model says p=0.8, it now gets ~80% accuracy
   - This allows for more informed confidence-based decisions
   - Crucial for safety applications where confidence matters

{'='*70}
PHASE 4 COMPLETED SUCCESSFULLY
{'='*70}
"""

print(report)

with open(OUTPUT_DIR / 'final_report.txt', 'w') as f:
    f.write(report)

print(f"\nReport saved to: {OUTPUT_DIR / 'final_report.txt'}")

## 13. Per-Class Calibration Analysis

In [ ]:
eval_df = pd.read_csv(OUTPUT_DIR / 'eval_detections.csv')

per_class_metrics = []

for cat in CONFIG['categories']:
    cat_data = eval_df[eval_df['category'] == cat]
    
    if len(cat_data) < 10:
        continue
    
    logits_cat = cat_data['logit'].values
    labels_cat = cat_data['is_tp'].values
    
    metrics_cat_before = compute_calibration_metrics(logits_cat, labels_cat, T=1.0)
    metrics_cat_after = compute_calibration_metrics(logits_cat, labels_cat, T=T_optimal)
    
    per_class_metrics.append({
        'category': cat,
        'n_samples': len(cat_data),
        'nll_before': metrics_cat_before['nll'],
        'nll_after': metrics_cat_after['nll'],
        'ece_before': metrics_cat_before['ece'],
        'ece_after': metrics_cat_after['ece'],
        'brier_before': metrics_cat_before['brier'],
        'brier_after': metrics_cat_after['brier']
    })

df_per_class = pd.DataFrame(per_class_metrics)
df_per_class['nll_improvement'] = df_per_class['nll_before'] - df_per_class['nll_after']
df_per_class['ece_improvement'] = df_per_class['ece_before'] - df_per_class['ece_after']

df_per_class.to_csv(OUTPUT_DIR / 'calibration_per_class.csv', index=False)

print("PER-CLASS CALIBRATION METRICS\n")
print(df_per_class.to_string(index=False))
print(f"\nSaved to: {OUTPUT_DIR / 'calibration_per_class.csv'}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cats = df_per_class['category'].values
x_pos = np.arange(len(cats))

# ECE
axes[0].bar(x_pos - 0.2, df_per_class['ece_before'], 0.4, label='Before', alpha=0.7, color='blue')
axes[0].bar(x_pos + 0.2, df_per_class['ece_after'], 0.4, label='After', alpha=0.7, color='red')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(cats, rotation=45, ha='right')
axes[0].set_ylabel('ECE', fontsize=12)
axes[0].set_title('ECE per Class', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# NLL
axes[1].bar(x_pos - 0.2, df_per_class['nll_before'], 0.4, label='Before', alpha=0.7, color='blue')
axes[1].bar(x_pos + 0.2, df_per_class['nll_after'], 0.4, label='After', alpha=0.7, color='red')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(cats, rotation=45, ha='right')
axes[1].set_ylabel('NLL', fontsize=12)
axes[1].set_title('NLL per Class', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'calibration_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {OUTPUT_DIR / 'calibration_per_class.png'}")

## 14. Verification Checklist

In [ ]:
print("="*70)
print("VERIFICATION CHECKLIST - DID CALIBRATION WORK?")
print("="*70)

with open(OUTPUT_DIR / 'calibration_metrics.json', 'r') as f:
    metrics = json.load(f)

T_opt = metrics['T_optimal']
nll_before = metrics['val_eval']['before']['nll']
nll_after = metrics['val_eval']['after']['nll']
ece_before = metrics['val_eval']['before']['ece']
ece_after = metrics['val_eval']['after']['ece']
brier_before = metrics['val_eval']['before']['brier']
brier_after = metrics['val_eval']['after']['brier']

checks = []

# Check 1: T is significantly different from 1
if abs(T_opt - 1.0) > 0.1:
    checks.append(("✓", f"T={T_opt:.3f} is different from 1.0 → There was imbalance"))
else:
    checks.append(("⚠️", f"T={T_opt:.3f} very close to 1.0 → Model was already calibrated"))

# Check 2: NLL improved
if nll_after < nll_before:
    improvement = ((nll_before - nll_after) / nll_before) * 100
    checks.append(("✓", f"NLL improved {improvement:.1f}% ({nll_before:.4f} → {nll_after:.4f})"))
else:
    checks.append(("❌", f"NLL did NOT improve ({nll_before:.4f} → {nll_after:.4f})"))

# Check 3: ECE improved
if ece_after < ece_before:
    improvement = ((ece_before - ece_after) / ece_before) * 100
    checks.append(("✓", f"ECE improved {improvement:.1f}% ({ece_before:.4f} → {ece_after:.4f})"))
else:
    checks.append(("❌", f"ECE did NOT improve ({ece_before:.4f} → {ece_after:.4f})"))

# Check 4: Brier improved
if brier_after < brier_before:
    improvement = ((brier_before - brier_after) / brier_before) * 100
    checks.append(("✓", f"Brier improved {improvement:.1f}% ({brier_before:.4f} → {brier_after:.4f})"))
else:
    checks.append(("❌", f"Brier did NOT improve ({brier_before:.4f} → {brier_after:.4f})"))

# Check 5: mAP maintained (should be almost the same)
if 'map_before' in dir() and 'map_after' in dir():
    map_diff = abs(map_after - map_before)
    if map_diff < 0.01:
        checks.append(("✓", f"mAP maintained ({map_before:.4f} → {map_after:.4f})"))
    else:
        checks.append(("⚠️", f"mAP changed significantly ({map_before:.4f} → {map_after:.4f})"))

print("\nRESULTS:")
print("-"*70)
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "="*70)

# Diagnosis if it didn't work
passed = sum(1 for s, _ in checks if s == "✓")
total = len(checks)

if passed >= 3:
    print("✅ CALIBRATION SUCCESSFUL")
    print("\nTemperature correctly adjusted probabilities.")
    print("Calibration metrics improved as expected.")
elif T_opt < 1.05 and T_opt > 0.95:
    print("⚠️  MODEL WAS ALREADY WELL CALIBRATED")
    print("\nT ≈ 1.0 indicates baseline model had good calibration.")
    print("This is POSITIVE: no need to apply temperature in production.")
else:
    print("❌ PROBLEM DETECTED")
    print("\nPossible causes:")
    print("1. Incorrect score→logit conversion (review inverse sigmoid function)")
    print("2. Dataset too small to optimize T")
    print("3. Very different distribution between val_calib and val_eval")
    print("\nReview:")
    print("- Diagnosis on val_calib (previous cell)")
    print("- Reliability diagrams (should show visual improvement)")
    print("- Score distribution (histograms)")

print("="*70)

## 15. ✅ Complete Final Verification

In [ ]:
print("="*80)
print("COMPLETE FINAL VERIFICATION - PHASE 4: TEMPERATURE SCALING")
print("="*80)

import json
import pandas as pd
import numpy as np

with open(OUTPUT_DIR / 'temperature.json', 'r') as f:
    temp_data = json.load(f)
with open(OUTPUT_DIR / 'calibration_metrics.json', 'r') as f:
    calib_metrics = json.load(f)

T_opt = temp_data['T_global']
nll_before = calib_metrics['val_eval']['before']['nll']
nll_after = calib_metrics['val_eval']['after']['nll']
ece_before = calib_metrics['val_eval']['before']['ece']
ece_after = calib_metrics['val_eval']['after']['ece']
brier_before = calib_metrics['val_eval']['before']['brier']
brier_after = calib_metrics['val_eval']['after']['brier']

calib_df = pd.read_csv(OUTPUT_DIR / 'calib_detections.csv')
eval_df = pd.read_csv(OUTPUT_DIR / 'eval_detections.csv')

print(f"\n{'1. OPTIMAL TEMPERATURE'}")
print("-"*80)
print(f"   T_optimal = {T_opt:.4f}")
if T_opt > 1.1:
    print(f"   ✓ T > 1.0 → Model was OVERCONFIDENT")
    print(f"   → Temperature Scaling REDUCES probabilities")
elif T_opt < 0.9:
    print(f"   ✓ T < 1.0 → Model was UNDERCONFIDENT")
else:
    print(f"   ⚠️  T ≈ 1.0 → Model was already calibrated")

print(f"\n{'2. PROCESSED DATA'}")
print("-"*80)
print(f"   val_calib: {len(calib_df):,} detections (TP: {calib_df['is_tp'].sum():,}, {calib_df['is_tp'].mean()*100:.1f}%)")
print(f"   val_eval:  {len(eval_df):,} detections (TP: {eval_df['is_tp'].sum():,}, {eval_df['is_tp'].mean()*100:.1f}%)")

print(f"\n{'3. CALIBRATION METRICS (val_eval)'}")
print("-"*80)
print(f"   {'Metric':<15} {'Before':<12} {'After':<12} {'Improv %':<12}")
print(f"   {'-'*15} {'-'*12} {'-'*12} {'-'*12}")

nll_pct = ((nll_before - nll_after) / nll_before) * 100
ece_pct = ((ece_before - ece_after) / ece_before) * 100
brier_pct = ((brier_before - brier_after) / brier_before) * 100

print(f"   {'NLL':<15} {nll_before:<12.4f} {nll_after:<12.4f} {nll_pct:>10.2f}%  {'✓' if nll_pct > 0 else '✗'}")
print(f"   {'ECE':<15} {ece_before:<12.4f} {ece_after:<12.4f} {ece_pct:>10.2f}%  {'✓' if ece_pct > 0 else '✗'}")
print(f"   {'Brier':<15} {brier_before:<12.4f} {brier_after:<12.4f} {brier_pct:>10.2f}%  {'✓' if brier_pct > 0 else '✗'}")

print(f"\n{'4. GENERATED ARTIFACTS'}")
print("-"*80)
artifacts = [
    'temperature.json', 'calib_detections.csv', 'eval_detections.csv',
    'calibration_metrics.json', 'reliability_diagram.png', 
    'confidence_distribution.png', 'risk_coverage.png',
    'temperature_per_class.json', 'calibration_per_class.csv', 
    'calibration_per_class.png', 'final_report.txt'
]
for artifact in artifacts:
    exists = (OUTPUT_DIR / artifact).exists()
    print(f"   {'✓' if exists else '✗'} {artifact}")

print(f"\n{'5. FINAL CHECKLIST'}")
print("-"*80)
checks_passed = 0
total_checks = 4

if abs(T_opt - 1.0) > 0.1:
    print(f"   ✓ T significantly ≠ 1.0 (T={T_opt:.3f})")
    checks_passed += 1
else:
    print(f"   ⚠️  T very close to 1.0 (T={T_opt:.3f})")

if nll_pct > 0:
    print(f"   ✓ NLL improved {nll_pct:.2f}%")
    checks_passed += 1
else:
    print(f"   ✗ NLL worsened {abs(nll_pct):.2f}%")

if ece_pct > 0:
    print(f"   ✓ ECE improved {ece_pct:.2f}%")
    checks_passed += 1
else:
    print(f"   ✗ ECE worsened {abs(ece_pct):.2f}%")

if brier_pct > 0:
    print(f"   ✓ Brier improved {brier_pct:.2f}%")
    checks_passed += 1
else:
    print(f"   ✗ Brier worsened {abs(brier_pct):.2f}%")

print("\n" + "="*80)
if checks_passed >= 3:
    print(f"✅ CALIBRATION SUCCESSFUL ({checks_passed}/{total_checks} checks passed)")
    print("\n   Temperature correctly adjusted probabilities.")
    print("   Calibration metrics improved significantly.")
    print(f"\n   💡 Use T={T_opt:.4f} in production for calibrated inference.")
elif checks_passed >= 2:
    print(f"⚠️  PARTIAL CALIBRATION ({checks_passed}/{total_checks} checks passed)")
    print("\n   Some metrics improved, review per-class results.")
elif T_opt > 0.95 and T_opt < 1.05:
    print(f"ℹ️  MODEL ALREADY CALIBRATED ({checks_passed}/{total_checks} checks)")
    print("\n   T ≈ 1.0 indicates baseline model had good calibration.")
else:
    print(f"❌ PROBLEM DETECTED ({checks_passed}/{total_checks} checks passed)")
    print("\n   Calibration did not work as expected. Review data and process.")

print("="*80)
print("\n📊 Important plots:")
print("   • reliability_diagram.png (key for visualizing calibration)")
print("   • confidence_distribution.png")
print("   • risk_coverage.png")
print("   • calibration_per_class.png")
print("\n📄 Complete reports:")
print("   • final_report.txt")
